In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/schema.json
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00135.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00218.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00150.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00021.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00075.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00036.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00167.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/

In [3]:
import pyarrow.parquet as pq

file_path = "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00000.parquet"

parquet_file = pq.ParquetFile(file_path)

print("Number of rows:", parquet_file.metadata.num_rows)

print("\nColumns:")
print(parquet_file.schema_arrow.names)

Number of rows: 25000

Columns:
['abstract', 'additional_entities', 'date_created', 'date_modified', 'description', 'event', 'identifier', 'image', 'in_language', 'infoboxes', 'is_part_of', 'license', 'main_entity', 'name', 'references', 'sections', 'tables', 'url', 'version']


In [5]:
# Search for an article by name

search_term = "ISRO"

matches = [
    name for name in names
    if search_term.lower() in str(name).lower()
]

print("Search results for:", search_term)
print()

if matches:
    for name in matches[:20]:
        print(name)
else:
    print("No matching article found.")

Search results for: ISRO

No matching article found.


In [4]:
import pyarrow.parquet as pq

table = pq.read_table(
    file_path,
    columns=["name"],
)

names = table["name"].to_pylist()

print("First 20 article names:")
for name in names[:20]:
    print(name)

First 20 article names:
"Khan gizi" spring
1113 in Italy
18th Infantry Division (France)
1933 Greek parliamentary election
1903 German football championship
1936 Ottawa municipal election
1709 in Scotland
1930 in Mandatory Palestine
1892 Rangitikei by-election
1912–13 Scottish Districts season
12E
12th International Emmy Awards
(+)-Caryolan-1-ol synthase
1690s in archaeology
10.5 cm kanon m/34
1912 Columbus Panhandles season
1929 in rail transport
HIghest organ of state power
10th Tactical Squadron
18th Infantry Brigade


In [6]:
import os
import pyarrow.parquet as pq

data_folder = "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data"

search_term = "ISRO"
found = []

files = sorted([
    os.path.join(data_folder, f)
    for f in os.listdir(data_folder)
    if f.endswith(".parquet")
])

print("Searching", len(files), "dataset files...")
print()

for file in files:
    parquet_file = pq.ParquetFile(file)
    
    for batch in parquet_file.iter_batches(
        columns=["name"],
        batch_size=5000
    ):
        batch_names = batch["name"].to_pylist()
        
        for name in batch_names:
            if search_term.lower() in str(name).lower():
                found.append((name, os.path.basename(file)))
        
        if found:
            break
    
    if found:
        break

print("Search results for:", search_term)
print()

if found:
    for name, filename in found[:20]:
        print("Article:", name)
        print("File:", filename)
else:
    print("No matching article found in the dataset.")

Searching 265 dataset files...

Search results for: ISRO

Article: Congregation Tiferes Yisroel
File: enwiki_namespace_0_00004.parquet


In [8]:
# Faster search for the article

import pyarrow.parquet as pq
import os

search_terms = [
    "ISRO",
    "Indian Space Research Organisation"
]

found = []

for file in files:
    parquet_file = pq.ParquetFile(file)

    # Read only the name column
    table = parquet_file.read(columns=["name"])
    names_in_file = table["name"]

    for i in range(len(names_in_file)):
        name = str(names_in_file[i].as_py()).strip()

        if any(name.lower() == term.lower() for term in search_terms):
            found.append((name, os.path.basename(file)))

    if found:
        break

print("Search completed.")
print()

if found:
    for name, filename in found:
        print("Article:", name)
        print("File:", filename)
else:
    print("Exact article not found in the searched files.")

Search completed.

Article: ISRO
File: enwiki_namespace_0_00234.parquet


In [9]:
# Retrieve the ISRO article from the dataset

article_file = os.path.join(
    data_folder,
    "enwiki_namespace_0_00234.parquet"
)

parquet_file = pq.ParquetFile(article_file)

article_data = None

for batch in parquet_file.iter_batches(
    columns=[
        "name",
        "description",
        "abstract",
        "main_entity",
        "additional_entities",
        "sections"
    ],
    batch_size=5000
):
    names_batch = batch["name"].to_pylist()

    for i, name in enumerate(names_batch):
        if str(name).strip().lower() == "isro":
            article_data = {
                "name": name,
                "description": batch["description"][i].as_py(),
                "abstract": batch["abstract"][i].as_py(),
                "main_entity": batch["main_entity"][i].as_py(),
                "additional_entities": batch["additional_entities"][i].as_py(),
                "sections": batch["sections"][i].as_py()
            }
            break

    if article_data:
        break

print("Article retrieved successfully!")
print()
print("Article Name:", article_data["name"])
print("Description:", article_data["description"])
print()
print("Abstract:")
print(article_data["abstract"])

Article retrieved successfully!

Article Name: ISRO
Description: Indian national space and aeronautics agency

Abstract:
The Indian Space Research Organisation is the national space agency of India, headquartered in Bengaluru, Karnataka. It serves as the principal research and development arm of the Department of Space (DoS), overseen by the Prime Minister of India, with the Chairman of ISRO also serving as the chief executive of the DoS. It is primarily responsible for space-based operations, space exploration, international space cooperation and the development of related technologies. The agency maintains a constellation of imaging, communications and remote sensing satellites. It operates the GAGAN and IRNSS satellite navigation systems. It has sent three missions to the Moon and one mission to Mars. Formerly, ISRO was known as the Indian National Committee for Space Research (INCOSPAR), which was set up in 1962 by Prime Minister Jawaharlal Nehru on the recommendation of scientist 

In [10]:
# Explore the related information available for ISRO

print("MAIN ENTITY")
print("================")
print(article_data["main_entity"])

print("\nADDITIONAL ENTITIES")
print("================")
print(article_data["additional_entities"])

print("\nSECTIONS")
print("================")
print(article_data["sections"])

MAIN ENTITY
{'identifier': 'Q229058', 'url': 'https://www.wikidata.org/entity/Q229058'}

ADDITIONAL ENTITIES
[{'aspects': ['CQR', 'D.en', 'O', 'S', 'T'], 'identifier': 'Q229058', 'url': 'https://www.wikidata.org/entity/Q229058'}, {'aspects': ['S'], 'identifier': 'Q7012369', 'url': 'https://www.wikidata.org/entity/Q7012369'}]

SECTIONS
[{"type":"section","name":"Abstract","has_parts":[{"type":"paragraph","value":"The Indian Space Research Organisation (ISRO / ˈ ɪ s r oʊ /) is the national space agency of India, headquartered in Bengaluru, Karnataka. It serves as the principal research and development arm of the Department of Space (DoS), overseen by the Prime Minister of India, with the Chairman of ISRO also serving as the chief executive of the DoS. It is primarily responsible for space-based operations, space exploration, international space cooperation and the development of related technologies. The agency maintains a constellation of imaging, communications and remote sensing satel

In [11]:
# Find related articles using the entity identifiers

entity_ids = set()

# Main entity
if article_data["main_entity"]:
    entity_ids.add(article_data["main_entity"]["identifier"])

# Additional entities
for entity in article_data["additional_entities"]:
    entity_ids.add(entity["identifier"])

print("Entity IDs found:")
for entity_id in entity_ids:
    print(entity_id)

print("\nSearching the dataset for articles connected to these entities...")

Entity IDs found:
Q7012369
Q229058

Searching the dataset for articles connected to these entities...


In [ ]:
# Search all English Wikipedia files for the related entity IDs

related_articles = []

for file in files:
    parquet_file = pq.ParquetFile(file)

    for batch in parquet_file.iter_batches(
        columns=["name", "main_entity", "additional_entities"],
        batch_size=2000
    ):
        names_batch = batch["name"].to_pylist()
        main_batch = batch["main_entity"].to_pylist()
        additional_batch = batch["additional_entities"].to_pylist()

        for i in range(len(names_batch)):
            matched = False

            # Check main entity
            main_entity = main_batch[i]

            if main_entity and main_entity.get("identifier") in entity_ids:
                matched = True

            # Check additional entities
            if not matched:
                additional_entities = additional_batch[i] or []

                for entity in additional_entities:
                    if entity.get("identifier") in entity_ids:
                        matched = True
                        break

            if matched:
                related_articles.append(names_batch[i])

        if len(related_articles) >= 30:
            break

    if len(related_articles) >= 30:
        break

print("Related articles found:", len(related_articles))
print()

for article in related_articles[:30]:
    print(article)

In [1]:
# Extract possible related information from ISRO sections

import json

sections = article_data["sections"]

print("Number of sections:", len(sections))
print()

for section in sections[:10]:
    print("Section:", section.get("name"))
    
    text = json.dumps(section)
    
    # Show article/entity-related information if present
    for key in ["links", "entities", "related_articles", "has_parts"]:
        if key in section:
            print(key, ":", section[key])
    
    print("-" * 60)

NameError: name 'article_data' is not defined

In [2]:
import os
import json
import pyarrow.parquet as pq

data_folder = "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data"

article_file = os.path.join(
    data_folder,
    "enwiki_namespace_0_00234.parquet"
)

print("ISRO file found:", os.path.exists(article_file))

ISRO file found: True


In [3]:
# Retrieve the ISRO article again

parquet_file = pq.ParquetFile(article_file)

article_data = None

for batch in parquet_file.iter_batches(
    columns=[
        "name",
        "description",
        "abstract",
        "main_entity",
        "additional_entities",
        "sections"
    ],
    batch_size=5000
):
    names_batch = batch["name"].to_pylist()

    for i, name in enumerate(names_batch):
        if str(name).strip().lower() == "isro":
            article_data = {
                "name": name,
                "description": batch["description"][i].as_py(),
                "abstract": batch["abstract"][i].as_py(),
                "main_entity": batch["main_entity"][i].as_py(),
                "additional_entities": batch["additional_entities"][i].as_py(),
                "sections": batch["sections"][i].as_py()
            }
            break

    if article_data:
        break

print("Article retrieved:", article_data["name"])

Article retrieved: ISRO


In [4]:
# Inspect ISRO section names

sections = article_data["sections"]

if isinstance(sections, str):
    sections = json.loads(sections)

print("Number of sections:", len(sections))
print()

for section in sections:
    print(section.get("name"))

Number of sections: 14

Abstract
History
Goals and objectives
Organisation structure and facilities
General satellite programmes
Launch vehicles
Human spaceflight programme
Planetary sciences and astronomy
Extraterrestrial exploration
Upcoming launches
Future projects
Applications
International cooperations
Corporate affairs


In [5]:
# Display the ISRO article in an organized way

print("=" * 70)
print("WIKIKNOWLEDGE EXPLORER")
print("=" * 70)

print("\nARTICLE:", article_data["name"])
print("DESCRIPTION:", article_data["description"])

print("\nSECTIONS AVAILABLE")
print("-" * 70)

for number, section in enumerate(sections, start=1):
    print(f"{number}. {section.get('name')}")

print("\nMAIN ENTITY")
print("-" * 70)
print("Identifier:", article_data["main_entity"]["identifier"])
print("URL:", article_data["main_entity"]["url"])

WIKIKNOWLEDGE EXPLORER

ARTICLE: ISRO
DESCRIPTION: Indian national space and aeronautics agency

SECTIONS AVAILABLE
----------------------------------------------------------------------
1. Abstract
2. History
3. Goals and objectives
4. Organisation structure and facilities
5. General satellite programmes
6. Launch vehicles
7. Human spaceflight programme
8. Planetary sciences and astronomy
9. Extraterrestrial exploration
10. Upcoming launches
11. Future projects
12. Applications
13. International cooperations
14. Corporate affairs

MAIN ENTITY
----------------------------------------------------------------------
Identifier: Q229058
URL: https://www.wikidata.org/entity/Q229058


In [6]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# Create section dropdown
section_names = [section.get("name") for section in sections]

section_dropdown = widgets.Dropdown(
    options=section_names,
    description="Section:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

output = widgets.Output()

def show_section(change):
    if change["name"] == "value":
        selected_name = change["new"]

        for section in sections:
            if section.get("name") == selected_name:
                with output:
                    output.clear_output()
                    print("=" * 70)
                    print(selected_name.upper())
                    print("=" * 70)
                    
                    for part in section.get("has_parts", []):
                        if part.get("type") == "paragraph":
                            print(part.get("value", ""))
                            print()

section_dropdown.observe(show_section)

display(section_dropdown)
display(output)

# Show the first section initially
show_section({"name": "value", "new": section_names[0]})

Dropdown(description='Section:', layout=Layout(width='500px'), options=('Abstract', 'History', 'Goals and obje…

Output()

In [7]:
# Check what information is stored inside a section

history_section = None

for section in sections:
    if section.get("name") == "History":
        history_section = section
        break

print("Keys inside the History section:")
print(history_section.keys())

print("\nFirst part of the History section:")
print(history_section.get("has_parts", [])[:2])

Keys inside the History section:
dict_keys(['type', 'name', 'has_parts'])

First part of the History section:
[{'type': 'section', 'name': 'Agency logo', 'has_parts': [{'type': 'paragraph', 'value': 'ISRO has an official logo since 2002. It consists of an orange arrow shooting upwards attached with two blue coloured satellite panels with the name of ISRO written in two sets of text, orange-coloured Devanagari on the left and blue-coloured English in the Prakrit typeface on the right.', 'links': [{'url': 'https://en.wikipedia.org/wiki/Devanagari', 'text': 'Devanagari'}, {'url': 'https://en.wikipedia.org/wiki/Prakrit', 'text': 'Prakrit'}, {'url': 'https://en.wikipedia.org/wiki/ISRO#cite_note-SpaceIndia_Q2_2002-22'}, {'url': 'https://en.wikipedia.org/wiki/ISRO#cite_note-logo-23'}], 'citations': [{'identifier': 'cite_note-22', 'text': '[19]'}, {'identifier': 'cite_note-23', 'text': '[20]'}]}]}, {'type': 'section', 'name': 'Formative years', 'has_parts': [{'type': 'paragraph', 'value': 'Mod

In [8]:
# Extract related Wikipedia articles from ISRO

related_info = []

def extract_links(parts):
    for part in parts:
        if part.get("type") == "paragraph":
            for link in part.get("links", []):
                if "text" in link:
                    related_info.append({
                        "name": link["text"],
                        "url": link["url"]
                    })

        elif part.get("type") == "section":
            extract_links(part.get("has_parts", []))


# Check every main section
for section in sections:
    extract_links(section.get("has_parts", []))


# Remove duplicate articles
unique_related = {}

for item in related_info:
    unique_related[item["name"]] = item["url"]

print("Related articles found:", len(unique_related))
print()

for name, url in list(unique_related.items())[:30]:
    print(name, "->", url)

Related articles found: 367

/ ˈ ɪ s r oʊ / -> https://en.wikipedia.org/wiki/Help:IPA/English
space agency -> https://en.wikipedia.org/wiki/List_of_government_space_agencies
Bengaluru -> https://en.wikipedia.org/wiki/Bengaluru
Department of Space -> https://en.wikipedia.org/wiki/Department_of_Space
Prime Minister of India -> https://en.wikipedia.org/wiki/Prime_Minister_of_India
Chairman of ISRO -> https://en.wikipedia.org/wiki/Chairperson_of_ISRO
space exploration -> https://en.wikipedia.org/wiki/Space_exploration
imaging -> https://en.wikipedia.org/wiki/Earth_observation_satellite
communications -> https://en.wikipedia.org/wiki/Communications_satellite
remote sensing -> https://en.wikipedia.org/wiki/Remote_sensing
GAGAN -> https://en.wikipedia.org/wiki/GPS-aided_GEO_augmented_navigation
IRNSS -> https://en.wikipedia.org/wiki/Indian_Regional_Navigation_Satellite_System
satellite navigation -> https://en.wikipedia.org/wiki/Satellite_navigation
three missions -> https://en.wikipedia.org/

In [9]:
# Organize related information

# Remove unwanted citation links and keep useful Wikipedia links
related_articles = []

for name, url in unique_related.items():
    if "#cite_note" not in url and "Help:IPA" not in url:
        related_articles.append({
            "name": name,
            "url": url
        })

print("Useful related articles:", len(related_articles))
print()

for i, article in enumerate(related_articles[:40], start=1):
    print(f"{i}. {article['name']}")

Useful related articles: 366

1. space agency
2. Bengaluru
3. Department of Space
4. Prime Minister of India
5. Chairman of ISRO
6. space exploration
7. imaging
8. communications
9. remote sensing
10. GAGAN
11. IRNSS
12. satellite navigation
13. three missions
14. Moon
15. one mission
16. Mars
17. Indian National Committee for Space Research
18. Jawaharlal Nehru
19. Vikram Sarabhai
20. Department of Atomic Energy
21. space technology
22. Aryabhata
23. Soviet
24. Interkosmos
25. RS-1
26. SLV-3
27. seventh country
28. small-lift
29. medium-lift launch vehicles
30. deep space
31. cryogenic engines
32. extraterrestrial missions
33. artificial satellites
34. unmanned landing
35. disaster management
36. telemedicine
37. ISRO's spin-off technologies
38. Devanagari
39. Prakrit
40. S. K. Mitra


In [10]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Keep useful related articles
display_articles = [
    article for article in related_articles
    if article["name"].lower() not in [
        "three missions",
        "one mission",
        "seventh country"
    ]
]

# Search box
search_box = widgets.Text(
    placeholder="Search related information...",
    description="Search:",
    layout=widgets.Layout(width="600px")
)

# Related article dropdown
article_dropdown = widgets.Dropdown(
    options=[article["name"] for article in display_articles[:50]],
    description="Explore:",
    layout=widgets.Layout(width="600px")
)

output = widgets.Output()

def show_related(change=None):
    selected = article_dropdown.value

    for article in display_articles:
        if article["name"] == selected:
            with output:
                output.clear_output()

                display(HTML(
                    f"""
                    <h3>Related Information</h3>
                    <p><b>Article:</b> {article["name"]}</p>
                    <p><b>Wikipedia Link:</b>
                    <a href="{article["url"]}" target="_blank">
                    Open Article
                    </a></p>
                    """
                ))
            break


def search_related(change):
    term = search_box.value.lower().strip()

    matches = [
        article["name"]
        for article in display_articles
        if term in article["name"].lower()
    ]

    article_dropdown.options = matches[:50]

    if matches:
        article_dropdown.value = matches[0]
    else:
        with output:
            output.clear_output()
            print("No related information found.")


search_box.observe(search_related, names="value")
article_dropdown.observe(show_related, names="value")

display(search_box)
display(article_dropdown)
display(output)

show_related()

Text(value='', description='Search:', layout=Layout(width='600px'), placeholder='Search related information...…

Dropdown(description='Explore:', layout=Layout(width='600px'), options=('space agency', 'Bengaluru', 'Departme…

Output()

In [11]:
# WikiKnowledge Explorer - Article Search

article_search = widgets.Text(
    placeholder="Enter article name, e.g. ISRO",
    description="Article:",
    layout=widgets.Layout(width="600px")
)

search_output = widgets.Output()

def search_article(change):
    term = article_search.value.strip().lower()

    if not term:
        return

    with search_output:
        search_output.clear_output()

        if term == "isro":
            print("Article found: ISRO")
            print()
            print("Description:", article_data["description"])
            print()
            print("Available Sections:")

            for section in sections:
                print("•", section.get("name"))

        else:
            print("Currently loaded demo article: ISRO")
            print("Try searching: ISRO")

article_search.observe(search_article, names="value")

display(article_search)
display(search_output)

Text(value='', description='Article:', layout=Layout(width='600px'), placeholder='Enter article name, e.g. ISR…

Output()

In [12]:
# ============================================
# WIKIKNOWLEDGE EXPLORER
# ============================================

from IPython.display import display, HTML
import ipywidgets as widgets

# ---------- TITLE ----------
display(HTML("""
<div style="
    background:#1f3c88;
    color:white;
    padding:20px;
    border-radius:12px;
    margin-bottom:15px;
">
    <h1 style="margin:0;">WikiKnowledge Explorer</h1>
    <p style="margin:6px 0 0;">
        Explore Wikipedia articles and discover related information
    </p>
</div>
"""))

# ---------- ARTICLE SEARCH ----------
article_box = widgets.Text(
    value="ISRO",
    placeholder="Enter article name",
    description="Article:",
    layout=widgets.Layout(width="650px")
)

search_button = widgets.Button(
    description="Search Article",
    button_style="primary",
    icon="search"
)

article_output = widgets.Output()

# ---------- SECTION SELECTOR ----------
section_dropdown = widgets.Dropdown(
    options=[section.get("name") for section in sections],
    description="Section:",
    layout=widgets.Layout(width="650px")
)

section_output = widgets.Output()

# ---------- RELATED SEARCH ----------
related_box = widgets.Text(
    placeholder="Search related information",
    description="Related:",
    layout=widgets.Layout(width="650px")
)

related_dropdown = widgets.Dropdown(
    options=[article["name"] for article in display_articles[:50]],
    description="Explore:",
    layout=widgets.Layout(width="650px")
)

related_output = widgets.Output()


# ---------- ARTICLE SEARCH FUNCTION ----------
def search_main_article(button):
    term = article_box.value.strip().lower()

    with article_output:
        article_output.clear_output()

        if term == "isro":
            print("✓ Article found")
            print()
            print("ARTICLE:", article_data["name"])
            print("DESCRIPTION:", article_data["description"])
            print()
            print("MAIN ENTITY:", article_data["main_entity"]["identifier"])
        else:
            print("Demo dataset article currently loaded: ISRO")
            print("Please search for ISRO.")


# ---------- SECTION FUNCTION ----------
def show_selected_section(change=None):
    selected = section_dropdown.value

    with section_output:
        section_output.clear_output()

        for section in sections:
            if section.get("name") == selected:

                print("=" * 70)
                print(selected.upper())
                print("=" * 70)
                print()

                for part in section.get("has_parts", []):

                    if part.get("type") == "paragraph":
                        print(part.get("value", ""))
                        print()

                    elif part.get("type") == "section":
                        print("\n" + part.get("name", ""))
                        print("-" * 50)

                        for subpart in part.get("has_parts", []):
                            if subpart.get("type") == "paragraph":
                                print(subpart.get("value", ""))
                                print()


# ---------- RELATED SEARCH FUNCTION ----------
def search_related(change):
    term = related_box.value.lower().strip()

    if not term:
        matches = display_articles[:50]
    else:
        matches = [
            article["name"]
            for article in display_articles
            if term in article["name"].lower()
        ][:50]

    related_dropdown.options = matches


# ---------- RELATED ARTICLE FUNCTION ----------
def show_related_article(change=None):
    selected = related_dropdown.value

    if not selected:
        return

    for article in display_articles:
        if article["name"] == selected:

            with related_output:
                related_output.clear_output()

                display(HTML(f"""
                <div style="
                    padding:15px;
                    border:1px solid #ddd;
                    border-radius:10px;
                    margin-top:10px;
                ">
                    <h3>{article["name"]}</h3>
                    <p>
                        This article is linked from the ISRO article
                        in the Wikimedia Structured Wikipedia Dataset.
                    </p>
                    <a href="{article["url"]}" target="_blank">
                        Open Wikipedia Article →
                    </a>
                </div>
                """))

            break


# ---------- CONNECT EVENTS ----------
search_button.on_click(search_main_article)

section_dropdown.observe(
    show_selected_section,
    names="value"
)

related_box.observe(
    search_related,
    names="value"
)

related_dropdown.observe(
    show_related_article,
    names="value"
)


# ---------- DISPLAY ----------
display(HTML("<h2>🔎 Search Article</h2>"))
display(article_box)
display(search_button)
display(article_output)

display(HTML("<h2>📑 Explore Article Sections</h2>"))
display(section_dropdown)
display(section_output)

display(HTML("<h2>🔗 Discover Related Information</h2>"))
display(related_box)
display(related_dropdown)
display(related_output)

# Show initial content
search_main_article(None)
show_selected_section()
show_related_article()

Text(value='ISRO', description='Article:', layout=Layout(width='650px'), placeholder='Enter article name')

Button(button_style='primary', description='Search Article', icon='search', style=ButtonStyle())

Output()

Dropdown(description='Section:', layout=Layout(width='650px'), options=('Abstract', 'History', 'Goals and obje…

Output()

Text(value='', description='Related:', layout=Layout(width='650px'), placeholder='Search related information')

Dropdown(description='Explore:', layout=Layout(width='650px'), options=('space agency', 'Bengaluru', 'Departme…

Output()

In [13]:
# ============================================
# CLEANER WIKIKNOWLEDGE EXPLORER UI
# ============================================

from IPython.display import display, HTML
import ipywidgets as widgets

# ---------- HEADER ----------
display(HTML("""
<div style="
    background: linear-gradient(135deg, #172554, #2563eb);
    color: white;
    padding: 25px;
    border-radius: 15px;
    margin-bottom: 20px;
">
    <h1 style="margin:0;">WikiKnowledge Explorer</h1>
    <p style="margin:8px 0 0;">
        Explore an article and discover connected information
        from the Wikimedia Structured Wikipedia Dataset
    </p>
</div>
"""))

# ---------- ARTICLE INFORMATION ----------
display(HTML("""
<h2>🔎 Article</h2>
"""))

display(HTML(f"""
<div style="
    background:#f8fafc;
    padding:18px;
    border-radius:12px;
    border:1px solid #dbeafe;
    margin-bottom:15px;
">
    <h2 style="margin-top:0;">{article_data["name"]}</h2>
    <p><b>Description:</b> {article_data["description"]}</p>
    <p><b>Main Entity:</b> {article_data["main_entity"]["identifier"]}</p>
</div>
"""))

# ---------- SECTION EXPLORER ----------
display(HTML("<h2>📑 Explore Sections</h2>"))

section_names = [section.get("name") for section in sections]

section_selector = widgets.Select(
    options=section_names,
    value=section_names[0],
    description="Sections:",
    rows=8,
    layout=widgets.Layout(width="650px"),
    style={"description_width": "80px"}
)

section_result = widgets.Output(
    layout=widgets.Layout(
        width="95%",
        max_height="450px",
        overflow="auto",
        border="1px solid #ddd",
        padding="15px"
    )
)

def display_section(change=None):

    selected = section_selector.value

    with section_result:
        section_result.clear_output()

        for section in sections:

            if section.get("name") == selected:

                display(HTML(
                    f"<h2>{selected}</h2>"
                ))

                for part in section.get("has_parts", []):

                    if part.get("type") == "paragraph":

                        print(part.get("value", ""))
                        print()

                    elif part.get("type") == "section":

                        display(HTML(
                            f"<h3>{part.get('name', '')}</h3>"
                        ))

                        for subpart in part.get("has_parts", []):

                            if subpart.get("type") == "paragraph":
                                print(subpart.get("value", ""))
                                print()

section_selector.observe(display_section, names="value")

display(section_selector)
display(section_result)

display_section()


# ---------- RELATED INFORMATION ----------
display(HTML("<h2>🔗 Discover Related Information</h2>"))

related_search = widgets.Combobox(
    placeholder="Type a related topic...",
    options=[article["name"] for article in display_articles],
    description="Search:",
    ensure_option=True,
    layout=widgets.Layout(width="650px"),
    style={"description_width": "80px"}
)

related_result = widgets.Output()

def show_related(change=None):

    selected = related_search.value

    if not selected:
        return

    for article in display_articles:

        if article["name"] == selected:

            with related_result:

                related_result.clear_output()

                display(HTML(f"""
                <div style="
                    background:#f8fafc;
                    padding:18px;
                    border-radius:12px;
                    border:1px solid #dbeafe;
                    margin-top:10px;
                ">
                    <h3>{article["name"]}</h3>

                    <p>
                    This article is linked from the ISRO article
                    in the Wikimedia Structured Wikipedia Dataset.
                    </p>

                    <a href="{article["url"]}" target="_blank">
                    🌐 Open Wikipedia Article →
                    </a>
                </div>
                """))

            break

related_search.observe(show_related, names="value")

display(related_search)
display(related_result)

Select(description='Sections:', layout=Layout(width='650px'), options=('Abstract', 'History', 'Goals and objec…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

Combobox(value='', description='Search:', ensure_option=True, layout=Layout(width='650px'), options=('space ag…

Output()

In [14]:
# ============================================================
#        WIKIKNOWLEDGE EXPLORER - PREMIUM UI
# ============================================================

from IPython.display import display, HTML
import ipywidgets as widgets

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

section_names = [section.get("name") for section in sections]

# Select useful related articles
clean_related = [
    article for article in display_articles
    if article["name"].strip()
    and len(article["name"].strip()) > 2
]

# ------------------------------------------------------------
# HEADER
# ------------------------------------------------------------

display(HTML("""
<style>

.wke-container {
    font-family: Arial, sans-serif;
    max-width: 1050px;
    margin: auto;
}

.wke-header {
    background: linear-gradient(135deg, #111827, #312e81, #4f46e5);
    color: white;
    padding: 32px;
    border-radius: 20px;
    margin-bottom: 22px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.18);
}

.wke-header h1 {
    font-size: 32px;
    margin: 0 0 8px 0;
}

.wke-header p {
    font-size: 15px;
    margin: 0;
    opacity: 0.85;
}

.wke-card {
    background: white;
    border: 1px solid #e5e7eb;
    border-radius: 16px;
    padding: 20px;
    margin-bottom: 18px;
    box-shadow: 0 5px 18px rgba(0,0,0,0.06);
}

.wke-card-title {
    font-size: 20px;
    font-weight: bold;
    color: #111827;
    margin-bottom: 14px;
}

.wke-stat-container {
    display: flex;
    gap: 15px;
    margin-bottom: 20px;
}

.wke-stat {
    flex: 1;
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 14px;
    padding: 18px;
}

.wke-stat-number {
    font-size: 25px;
    font-weight: bold;
    color: #4338ca;
}

.wke-stat-label {
    font-size: 13px;
    color: #64748b;
    margin-top: 4px;
}

.wke-info {
    background: #eef2ff;
    border-left: 5px solid #4f46e5;
    padding: 15px;
    border-radius: 10px;
    color: #1e293b;
}

.wke-related {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    padding: 16px;
    border-radius: 12px;
}

.wke-link {
    display: inline-block;
    background: #4f46e5;
    color: white !important;
    padding: 9px 16px;
    border-radius: 9px;
    text-decoration: none !important;
    margin-top: 8px;
}

</style>

<div class="wke-container">

<div class="wke-header">
    <h1>🔭 WikiKnowledge Explorer</h1>
    <p>
        Explore Wikipedia knowledge, discover connected topics,
        and navigate through related information.
    </p>
</div>

</div>
"""))

# ------------------------------------------------------------
# ARTICLE CARD
# ------------------------------------------------------------

display(HTML(f"""
<div class="wke-container">

<div class="wke-card">

<div class="wke-card-title">
📄 Article Overview
</div>

<div class="wke-info">
    <h2 style="margin-top:0;">{article_data["name"]}</h2>
    <p style="margin-bottom:0;">
        {article_data["description"]}
    </p>
</div>

<br>

<div class="wke-stat-container">

<div class="wke-stat">
    <div class="wke-stat-number">{len(section_names)}</div>
    <div class="wke-stat-label">Article Sections</div>
</div>

<div class="wke-stat">
    <div class="wke-stat-number">{len(clean_related)}</div>
    <div class="wke-stat-label">Related Connections</div>
</div>

<div class="wke-stat">
    <div class="wke-stat-number">{article_data["main_entity"]["identifier"]}</div>
    <div class="wke-stat-label">Main Entity</div>
</div>

</div>

</div>

</div>
"""))

# ------------------------------------------------------------
# SECTION EXPLORER
# ------------------------------------------------------------

display(HTML("""
<div class="wke-container">
<div class="wke-card">

<div class="wke-card-title">
📚 Explore Article
</div>

<p style="color:#64748b;">
Select a section to explore the structured content of the article.
</p>

</div>
</div>
"""))

section_selector = widgets.Dropdown(
    options=section_names,
    value=section_names[0],
    description="Section",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "70px"}
)

section_output = widgets.Output(
    layout=widgets.Layout(
        width="95%",
        max_height="400px",
        overflow="auto",
        padding="20px",
        border="1px solid #e5e7eb"
    )
)

def show_section(change=None):

    selected = section_selector.value

    with section_output:
        section_output.clear_output()

        for section in sections:

            if section.get("name") == selected:

                display(HTML(
                    f"""
                    <h2 style="color:#312e81;">
                    📖 {selected}
                    </h2>
                    """
                ))

                for part in section.get("has_parts", []):

                    if part.get("type") == "paragraph":

                        print(part.get("value", ""))
                        print()

                    elif part.get("type") == "section":

                        display(HTML(
                            f"""
                            <h3 style="color:#4338ca;">
                            {part.get("name", "")}
                            </h3>
                            """
                        ))

                        for subpart in part.get("has_parts", []):

                            if subpart.get("type") == "paragraph":
                                print(subpart.get("value", ""))
                                print()

section_selector.observe(show_section, names="value")

display(section_selector)
display(section_output)

show_section()

# ------------------------------------------------------------
# RELATED INFORMATION
# ------------------------------------------------------------

display(HTML("""
<div class="wke-container">

<div class="wke-card">

<div class="wke-card-title">
🔗 Discover Related Information
</div>

<p style="color:#64748b;">
Search topics connected to the selected Wikipedia article.
</p>

</div>

</div>
"""))

related_search = widgets.Combobox(
    placeholder="Type a topic, person, place or organization...",
    options=[article["name"] for article in clean_related],
    description="Search",
    ensure_option=True,
    layout=widgets.Layout(width="700px"),
    style={"description_width": "70px"}
)

related_output = widgets.Output()

def show_related(change=None):

    selected = related_search.value

    if not selected:
        return

    for article in clean_related:

        if article["name"] == selected:

            with related_output:

                related_output.clear_output()

                display(HTML(f"""
                <div class="wke-related">

                    <h2 style="margin-top:0; color:#312e81;">
                        🔗 {article["name"]}
                    </h2>

                    <p style="color:#475569;">
                        This connection is present in the
                        Wikimedia Structured Wikipedia Dataset.
                    </p>

                    <a class="wke-link"
                       href="{article["url"]}"
                       target="_blank">
                       🌐 Open Wikipedia Article →
                    </a>

                </div>
                """))

            break

related_search.observe(show_related, names="value")

display(related_search)
display(related_output)

# ------------------------------------------------------------
# FOOTER
# ------------------------------------------------------------

display(HTML("""
<div class="wke-container">

<div style="
    text-align:center;
    padding:22px;
    color:#64748b;
    font-size:13px;
">
    WikiKnowledge Explorer • Wikimedia Structured Wikipedia Dataset
</div>

</div>
"""))

Dropdown(description='Section', layout=Layout(width='700px'), options=('Abstract', 'History', 'Goals and objec…

Output(layout=Layout(border_bottom='1px solid #e5e7eb', border_left='1px solid #e5e7eb', border_right='1px sol…

Combobox(value='', description='Search', ensure_option=True, layout=Layout(width='700px'), options=('space age…

Output()